In [6]:
# YOSEMITE DATA
import glob
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, r2_score
from sklearn.linear_model import LinearRegression, RidgeCV

def load_yosemite_dataframe():
    txt_files = sorted(glob.glob('yosemite-temperatures/yosemite_village/*.txt'))

    cols = [
        'WBANNO','UTC_DATE','UTC_TIME','LST_DATE','LST_TIME','CRX_VN','LONGITUDE','LATITUDE',
        'AIR_TEMPERATURE','PRECIPITATION','SOLAR_RADIATION','SR_FLAG','SURFACE_TEMPERATURE','ST_TYPE',
        'ST_FLAG','RELATIVE_HUMIDITY','RH_FLAG','SOIL_MOISTURE_5','SOIL_TEMPERATURE_5','WETNESS',
        'WET_FLAG','WIND_1_5','WIND_FLAG'
    ]

    frames = []
    for f in txt_files:
        dfi = pd.read_csv(f, sep=r'\s+', engine='python', header=None, names=cols)
        frames.append(dfi)

    df = pd.concat(frames, ignore_index=True)
    print(f'Loaded raw Yosemite files: {len(txt_files)} files')
    return df

yosemite = load_yosemite_dataframe()

feature_cols = [
    'AIR_TEMPERATURE',
    'RELATIVE_HUMIDITY',
    'SOLAR_RADIATION',
    'SURFACE_TEMPERATURE',
    'WIND_1_5',
]

df = yosemite[feature_cols + ['LST_DATE', 'LST_TIME']].copy()
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df[(df[feature_cols].notna().all(axis=1)) & ((df[feature_cols] > -900).all(axis=1))]

date_str = df['LST_DATE'].astype(str).str.replace('.0', '', regex=False)
parsed = pd.to_datetime(date_str, format='%Y%m%d', errors='coerce')
time_str = df['LST_TIME'].astype(str).str.replace('.0', '', regex=False).str.zfill(4)
hour = pd.to_numeric(time_str.str[:2], errors='coerce')

df = df[parsed.notna()].copy()
df['month'] = parsed[parsed.notna()].dt.month.values
df['hour'] = hour[parsed.notna()].values
df = df[df['hour'].between(0, 23)]

# Downsample for tractable runtime.
MAX_ROWS = 20000
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42)

X = df[feature_cols + ['hour']].values
y = df['month'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'linear': SVC(kernel='linear', C=1, random_state=42),
    'quadratic': SVC(kernel='poly', degree=2, coef0=1, C=1, gamma='scale', random_state=42),
    'rbf': SVC(kernel='rbf', C=1, gamma='scale', random_state=42),
}

svm_results = []
best_name, best_acc = None, -1
for name, svc in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('svc', svc)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    svm_results.append({'kernel': name, 'test_accuracy': acc})
    if acc > best_acc:
        best_name, best_acc = name, acc

svm_df = pd.DataFrame(svm_results).sort_values('test_accuracy', ascending=False)
print('SVM month-classification using weather features:')
print(svm_df.to_string(index=False))
print(f'Best SVM accuracy: {best_acc:.4f} ({best_name} kernel)')


Loaded raw Yosemite files: 6 files
SVM month-classification using weather features:
   kernel  test_accuracy
      rbf         0.3445
quadratic         0.3225
   linear         0.2660
Best SVM accuracy: 0.3445 (rbf kernel)


In [7]:
lin = LinearRegression()
lin.fit(X_train, y_train)
lin_r2 = r2_score(y_test, lin.predict(X_test))

print(f'Linear model R^2: {lin_r2:.4f}')


Linear model R^2: 0.0246


In [11]:
svr = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf', C=30, gamma='scale', epsilon=0.05))])
svr.fit(X_train, y_train)
svr_r2 = r2_score(y_test, svr.predict(X_test))

print(f'SVR R^2: {svr_r2:.4f}')


SVR R^2: 0.0901


In [9]:
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]))
])
ridge.fit(X_train, y_train)
ridge_r2 = r2_score(y_test, ridge.predict(X_test))

best_alpha = ridge.named_steps['ridge'].alpha_
print(f'Ridge (L2-penalized) R^2: {ridge_r2:.4f} | best alpha={best_alpha}')
if ridge_r2 > lin_r2:
    print('Ridge outperforms the linear model on R^2 for this split.')
elif ridge_r2 < lin_r2:
    print('Linear model outperforms Ridge on R^2 for this split.')
else:
    print('Ridge and linear model tie on R^2 for this split.')


Ridge (L2-penalized) R^2: 0.0247 | best alpha=10.0
Ridge outperforms the linear model on R^2 for this split.


In [12]:
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error

eval_models = {
    'linear': lin,
    'svr': svr,
    'ridge': ridge,
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_estimators = {
    'linear': LinearRegression(),
    'svr': Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf', C=30, gamma='scale', epsilon=0.05))]),
    'ridge': Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]))]),
}

rows = []
for name, model in eval_models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    cv_r2 = cross_val_score(clone(cv_estimators[name]), X, y, cv=cv, scoring='r2')

    rows.append({
        'model': name,
        'train_r2': r2_score(y_train, train_pred),
        'test_r2': r2_score(y_test, test_pred),
        'train_mae': mean_absolute_error(y_train, train_pred),
        'test_mae': mean_absolute_error(y_test, test_pred),
        'cv_r2_mean': cv_r2.mean(),
        'cv_r2_std': cv_r2.std(),
    })

eval_df = pd.DataFrame(rows).sort_values('test_r2', ascending=False)
print('Regression evaluation summary:')
print(eval_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))


Regression evaluation summary:
 model  train_r2  test_r2  train_mae  test_mae  cv_r2_mean  cv_r2_std
   svr    0.1248   0.0901     2.3868    2.4721      0.0773     0.0260
 ridge    0.0246   0.0247     2.9612    2.9618      0.0238     0.0010
linear    0.0246   0.0246     2.9611    2.9616      0.0238     0.0010
